# `matrix_function_forward_sum` — Interactive Visualization

**Pipeline:** `X @ W → N → σ(N) → S → sum(S) = L`

This is a **forward pass** that collapses a whole matrix down to a single scalar loss `L`.  
Every step is shown as a heatmap, and the final aggregation is shown as a bar chart so you can see which elements of `S` contribute most to `L`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display

def sigmoid(x):    return 1 / (1 + np.exp(-x))
def relu(x):       return np.maximum(0, x)
def tanh_(x):      return np.tanh(x)
def leaky_relu(x): return np.where(x > 0, x, 0.01 * x)
def linear(x):     return x

SIGMAS = {"sigmoid": sigmoid, "tanh": tanh_, "relu": relu,
          "leaky_relu": leaky_relu, "linear": linear}

def deriv(func, input_, delta=0.001):
    return (func(input_ + delta) - func(input_ - delta)) / (2 * delta)

def matrix_function_forward_sum(X, W, sigma):
    assert X.shape[1] == W.shape[0]
    N = X @ W
    S = sigma(N)
    L = np.sum(S)
    return N, S, L

print("Setup complete.")

Setup complete.


In [2]:
def heatmap(ax, data, title, fmt=".2f", cmap="RdBu_r", center=None):
    vmax = np.abs(data).max() or 1.0
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax) if center == 0 else None
    im = ax.imshow(data, cmap=cmap, norm=norm, aspect="auto", interpolation="nearest")
    plt.colorbar(im, ax=ax, shrink=0.8)
    rows, cols = data.shape
    for r in range(rows):
        for c in range(cols):
            val = data[r, c]
            color = "white" if abs(val) > 0.6 * vmax else "black"
            ax.text(c, r, f"{val:{fmt}}", ha="center", va="center",
                    fontsize=max(5, 9 - max(rows, cols)), color=color)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xticks(range(cols)); ax.set_yticks(range(rows))
    ax.set_xticklabels([f"c{i}" for i in range(cols)], fontsize=8)
    ax.set_yticklabels([f"r{i}" for i in range(rows)], fontsize=8)

def draw(batch, in_feat, out_feat, sigma_name, seed):
    sigma = SIGMAS[sigma_name]
    rng   = np.random.default_rng(seed)
    X     = rng.standard_normal((batch, in_feat))
    W     = rng.standard_normal((in_feat, out_feat))

    N, S, L = matrix_function_forward_sum(X, W, sigma)

    # σ curve data
    N_flat = N.flatten()
    S_flat = S.flatten()
    pad    = max(2.5, np.abs(N_flat).max() * 0.4)
    xs     = np.linspace(N_flat.min() - pad, N_flat.max() + pad, 500)
    ys     = sigma(xs)

    fig = plt.figure(figsize=(16, 16))
    fig.suptitle(
        f"matrix_function_forward_sum  |  σ={sigma_name}  |  "
        f"X:{X.shape}  W:{W.shape}  →  L = {L:.4f}",
        fontsize=13, fontweight="bold", y=0.995
    )
    gs = gridspec.GridSpec(4, 4, figure=fig, hspace=0.65, wspace=0.45)

    # ── Row 0: inputs ─────────────────────────────────────────────────────
    heatmap(fig.add_subplot(gs[0, 0:2]), X, f"X  {X.shape}\n(input)", center=0)
    heatmap(fig.add_subplot(gs[0, 2:4]), W, f"W  {W.shape}\n(weights)", center=0)

    # ── Row 1: forward steps ──────────────────────────────────────────────
    heatmap(fig.add_subplot(gs[1, 0:2]), N, f"N = X @ W  {N.shape}\n(pre-activation)", center=0)
    heatmap(fig.add_subplot(gs[1, 2:4]), S, f"S = σ(N)  {S.shape}\n(post-activation)")

    # ── Row 2: aggregation bar + scalar display ───────────────────────────
    ax_bar = fig.add_subplot(gs[2, 0:3])
    ax_L   = fig.add_subplot(gs[2, 3])

    # colour each bar by its value (positive=blue, negative=red)
    colours = ["#2563eb" if v >= 0 else "#dc2626" for v in S_flat]
    labels  = [f"S[{r},{c}]" for r in range(S.shape[0]) for c in range(S.shape[1])]
    bars = ax_bar.bar(range(len(S_flat)), S_flat, color=colours, edgecolor="white", linewidth=0.5)
    ax_bar.axhline(0, color="black", lw=0.8)
    ax_bar.axhline(L / len(S_flat), color="orange", lw=1.5, ls="--",
                   label=f"mean = {L/len(S_flat):.3f}")
    ax_bar.set_xticks(range(len(S_flat)))
    ax_bar.set_xticklabels(labels, rotation=60, fontsize=7, ha="right")
    ax_bar.set_ylabel("σ(N) value", fontsize=9)
    ax_bar.set_title("S = σ(N) — all elements (bars sum to L)", fontsize=11, fontweight="bold")
    ax_bar.legend(fontsize=9)
    ax_bar.grid(axis="y", alpha=0.3)
    pos_patch = mpatches.Patch(color="#2563eb", label="positive")
    neg_patch = mpatches.Patch(color="#dc2626", label="negative")
    ax_bar.legend(handles=[pos_patch, neg_patch,
                            mpatches.Patch(color="orange", label=f"mean={L/len(S_flat):.3f}")],
                  fontsize=8, loc="upper right")

    # Scalar L display
    ax_L.axis("off")
    color = "#16a34a" if L >= 0 else "#dc2626"
    ax_L.text(0.5, 0.6, f"L = sum(S)", ha="center", va="center", fontsize=12,
              color="gray")
    ax_L.text(0.5, 0.35, f"{L:.4f}", ha="center", va="center",
              fontsize=28, fontweight="bold", color=color,
              bbox=dict(boxstyle="round,pad=0.4", fc="#f0fdf4" if L >= 0 else "#fef2f2",
                        ec=color, lw=2))
    ax_L.set_title("Scalar output L", fontsize=11, fontweight="bold")

    # ── Row 3: σ curve + N values ─────────────────────────────────────────
    ax_s  = fig.add_subplot(gs[3, 0:2])
    ax_ds = fig.add_subplot(gs[3, 2:4])

    ax_s.plot(xs, ys, color="#2563eb", lw=2.5, label=f"σ(x)  [{sigma_name}]")
    ax_s.scatter(N_flat, S_flat, color="#dc2626", s=55, zorder=5,
                 label=f"(N, S) pairs  n={len(N_flat)}")
    for xv, yv in zip(N_flat, S_flat):
        ax_s.vlines(xv, 0, yv, color="#dc2626", lw=0.7, alpha=0.35, ls="--")
    ax_s.set_title(f"σ(x) — activation  [{sigma_name}]", fontsize=11, fontweight="bold")
    ax_s.set_xlabel("x  (N values)", fontsize=9); ax_s.set_ylabel("σ(x)  =  S values", fontsize=9)
    ax_s.legend(fontsize=9); ax_s.grid(True, alpha=0.3)

    dys    = deriv(sigma, xs)
    dy_pts = deriv(sigma, N_flat)
    ax_ds.plot(xs, dys, color="#16a34a", lw=2.5, label="σ'(x)")
    ax_ds.axhline(0, color="#9ca3af", lw=0.8, ls="--")
    ax_ds.scatter(N_flat, dy_pts, color="#dc2626", s=55, zorder=5,
                  label="σ'(N)  (gradient signal)")
    for xv, yv in zip(N_flat, dy_pts):
        ax_ds.vlines(xv, 0, yv, color="#dc2626", lw=0.7, alpha=0.35, ls="--")
    ax_ds.set_title("σ'(x) — derivative", fontsize=11, fontweight="bold")
    ax_ds.set_xlabel("x  (N values)", fontsize=9); ax_ds.set_ylabel("σ'(x)", fontsize=9)
    ax_ds.legend(fontsize=9); ax_ds.grid(True, alpha=0.3)

    plt.show()
    print(f"X:{X.shape}  W:{W.shape}  N:{N.shape}  S:{S.shape}  L={L:.6f}")

print("Draw function ready.")

Draw function ready.


In [3]:
w_batch    = widgets.IntSlider(value=3, min=1, max=8, description="batch (rows X)",
                               style={"description_width":"155px"}, layout=widgets.Layout(width="430px"))
w_in_feat  = widgets.IntSlider(value=4, min=1, max=8, description="in_feat (cols X = rows W)",
                               style={"description_width":"155px"}, layout=widgets.Layout(width="430px"))
w_out_feat = widgets.IntSlider(value=2, min=1, max=8, description="out_feat (cols W)",
                               style={"description_width":"155px"}, layout=widgets.Layout(width="430px"))
w_sigma    = widgets.Dropdown(options=list(SIGMAS.keys()), value="sigmoid",
                              description="σ (activation)", style={"description_width":"155px"})
w_seed     = widgets.IntSlider(value=42, min=0, max=999, description="random seed",
                               style={"description_width":"155px"}, layout=widgets.Layout(width="430px"))

ui = widgets.VBox([
    widgets.HTML("<b>Matrix dimensions</b>"),
    w_batch, w_in_feat, w_out_feat,
    widgets.HTML("<b>Activation &amp; randomness</b>"),
    w_sigma, w_seed,
])

out = widgets.interactive_output(
    draw,
    {"batch": w_batch, "in_feat": w_in_feat, "out_feat": w_out_feat,
     "sigma_name": w_sigma, "seed": w_seed}
)

display(ui, out)

Output()